In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [3]:
from pathlib import Path

import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm

from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]


In [5]:
samples_src = BranchDigraphDataset.discover_paths(RAW, TOPO, AV, dataset_name=list(DATASETS_PATH.keys()))

In [14]:
from fundus_vessels_toolkit.utils.profiling import Profiler

ID = 150

with Profiler():
    sample_info = samples_src[ID].process(
        output_dir=Path("tmp/test_data_process/"), resize_to=1024, mask_optic_disc=True, overwrite=True
    )
    print(f"Sample [{ID}]: {sample_info.dataset}/{sample_info.name}")
    sample = sample_info.load(load_av_maps=True)

Sample [150]: MAPLES-DR/20060530_53540_0100_PP
2


]8;id=78557;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:446\SampleSource.process]8;;\                    2.8s                    (runs=1)
├── ]8;id=455476;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:448\Load fundus image]8;;\                  ↳3.3%   92.8ms           (runs=1)
│   ├── ]8;id=807118;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:449\Read image from disk]8;;\           0.40%    ↳ 12%   11.4ms  (runs=1)
│   ├── ]8;id=243327;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:452\Crop to ROI & resize]8;;\           2.73%    ↳ 84%   77.6ms  (runs=1)
│   └── ]8;id=731872;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:468\Write processed fundus image]8;;\   0.13%    ↳4.0%    3.7ms  (runs=1)
├── ]8;id=101017;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:473\Load or compute OD and Macula]8;;\      ↳6.2%  175.5ms           (runs=1)
│   └── ]8;id=158667;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:388\Segment OD and Macula]8;;\          4.25%    ↳ 69%  120.8ms  (runs=1)
├── ]8;id=999374;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:486\Load and preprocess graphes]8;;\        ↳ 67%     1.9s           (runs=1)
│   ├── ]8;id=321489;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:525\Load AV map]8;;\                      12%    ↳ 18%  350.7ms  (runs=4, avg=  87.7ms)
│   ├── ]8;id=450944;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:534\av2tree.to_vgraph]8;;\                35%    ↳ 52%     1.0s  (runs=4, avg= 250.7ms)
│   ├── ]8;id=233634;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:538\Clean graph]8;;\                    1.32%    ↳2.0%   37.5ms  (runs=4, avg=   9.4ms)
│   ├── ]8;id=493786;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:545\Check graph]8;;\                    0.69‰    ↳0.1%    2.0ms  (runs=4, avg= 489.8µs)
│   └── ]8;id=587063;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:552\Save processed graphes]8;;\         0.36%    ↳0.5%   10.1ms  (runs=4, avg=   2.5ms)
└── ]8;id=12210;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:562\Load and preprocess GT topology]8;;\    ↳ 23%  659.0ms           (runs=1)
    ├── ]8;id=873974;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:566\VTree.load]8;;\                     0.43%    ↳1.9%   12.3ms  (runs=2, avg=   6.2ms)
    ├── ]8;id=802663;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:571\Transform VTree]8;;\                0.89%    ↳3.8%   25.4ms  (runs=2, avg=  12.7ms)
    ├── ]8;id=329274;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:575\Clean]8;;\                          0.28%    ↳1.2%    8.1ms  (runs=2, avg=   4.0ms)
    ├── ]8;id=329251;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:586\Check common branches]8;;\          0.67%    ↳2.9%   19.1ms  (runs=1)
    ├── ]8;id=805102;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:596\Rasterize topologies]8;;\             16%    ↳ 67%  441.1ms  (runs=1)
    └── ]8;id=838717;file:///home/gaby/Lab/Src/fund

In [15]:
sample.show(["gt", "fvt", "automorph", "vascx"])

GridBox(children=(HTML(value='<h3 style="text-align: center;">gt</h3>'), HTML(value='<h3 style="text-align: ce…

In [8]:
print(f"od_macula: {sample.fundus.infer_scale('od_macula'):.2f} μm/px")
print(f"od_diameter: {sample.fundus.infer_scale('od_diameter'):.2f} μm/px")
print(f"width: {sample.fundus.infer_scale('width'):.2f} μm/px")

print(f"Estimated max calibre: {200 / sample.fundus.scale:.2f} px")

od_macula: 12.46 μm/px
od_diameter: 10.39 μm/px
width: 14.93 μm/px
Estimated max calibre: 16.06 px


In [9]:
max_calibres = [
    max([c.data.max() for c in g.geometric_data().branch_data("CALIBRES") if len(c.data)])
    for g in sample.graphes.values()
]
for name, calib in zip(sample.graphes.keys(), max_calibres):
    print(f"{name}: max calibre = {calib:.2f} px, {calib * sample.fundus.scale:.2f} μm")

fvt: max calibre = 18.14 px, 226.01 μm
automorph: max calibre = 11.87 px, 147.92 μm
gt: max calibre = 14.17 px, 176.54 μm
vascx: max calibre = 13.70 px, 170.66 μm
